In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

../data/public\sample_submission.csv
../data/public\train.csv
../data/public\test\00755AT0HO.jpg
../data/public\test\00JDJCJDID.jpg
../data/public\test\013R8BTZDO.jpg
../data/public\test\01FR6RG5CM.jpg
../data/public\test\02BG8UIHZU.jpg
../data/public\test\02US33J1G6.jpg
../data/public\test\03H0T3NNFA.jpg
../data/public\test\03IEDG0ILA.jpg
../data/public\test\04K05CCESG.jpg
../data/public\test\04SH5KI2W4.jpg
../data/public\test\04TC8ECIEO.jpg
../data/public\test\05N9ITZMS3.jpg
../data/public\test\05W97C0WU1.jpg
../data/public\test\06WIRY4IAR.jpg
../data/public\test\078QET9SQY.jpg
../data/public\test\07TC60ELJB.jpg
../data/public\test\07WJQYZZ0G.jpg
../data/public\test\082EZD2231.jpg
../data/public\test\08FD4ULRYO.jpg
../data/public\test\08H3LAJ8OU.jpg
../data/public\test\08U96AUGLJ.jpg
../data/public\test\094V3Y2ASK.jpg
../data/public\test\096L3DTQKA.jpg
../data/public\test\0AK0RX05W9.jpg
../data/public\test\0ATN06ZB48.jpg
../data/public\test\0BC8ORHR01.jpg
../data/public\test\0BH62UWF

c:\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

import os
import pandas as pd

DATA_DIR = "/kaggle/input/competitions/industrial-surface-defect-detection/public"
sample_path = os.path.join(DATA_DIR, "sample_submission.csv")
output_path = "/kaggle/working/submission.csv"

submission = pd.read_csv(sample_path, keep_default_na=False)

# Predict no defect for every image-category combination
submission["mask_rle"] = " "

# Preserve the official submission format
submission = submission[["ID", "image_id", "defect_type", "mask_rle"]]
submission.to_csv(output_path, index=False, na_rep=" ")

# Verify the generated file
check = pd.read_csv(output_path, keep_default_na=False)
print(check.head())
print("Null values:", check.isnull().sum().sum())
print("Empty masks:", (check["mask_rle"].str.strip() == "").sum())
print("Saved to:", output_path)


   ID    image_id defect_type mask_rle
0   1  WAFNKCRR7Q    defect_a         
1   2  WAFNKCRR7Q    defect_b         
2   3  WAFNKCRR7Q    defect_c         
3   4  WAFNKCRR7Q    defect_d         
4   5  YZXKEWSUDD    defect_a         
Null values: 0
Empty masks: 12000
Saved to: /kaggle/working/submission.csv


In [3]:


import os
import re
import cv2
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from scipy import ndimage

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

DATA_ROOT = Path("/kaggle/input/competitions/industrial-surface-defect-detection/public")
TRAIN_DIR = DATA_ROOT / "train"
TEST_DIR = DATA_ROOT / "test"
TRAIN_CSV = DATA_ROOT / "train.csv"

train = pd.read_csv(TRAIN_CSV, keep_default_na=False)
train["mask_rle"] = train["mask_rle"].astype(str).str.strip()
train["image_id"] = train["image_id"].astype(str).str.strip()
train["defect_type"] = train["defect_type"].astype(str).str.strip()

CLASSES = ["defect_a", "defect_b", "defect_c", "defect_d"]
HEIGHT, WIDTH = 128, 800
N_PIXELS = HEIGHT * WIDTH

assert set(CLASSES).issubset(set(train["defect_type"]))
assert train["image_id"].nunique() == 11393
assert len(train) == 45572
assert train.groupby("image_id")["defect_type"].nunique().eq(4).all()

def decode_rle(rle, height=HEIGHT, width=WIDTH):
    mask = np.zeros(height * width, dtype=np.uint8)
    if not rle.strip():
        return mask.reshape((height, width), order="F")
    values = np.fromstring(rle, sep=" ", dtype=np.int64)
    if len(values) % 2:
        raise ValueError(f"RLE has an odd number of values: {rle[:80]}")
    starts = values[0::2] - 1
    lengths = values[1::2]
    for start, length in zip(starts, lengths):
        if length < 0:
            raise ValueError("Negative RLE run length")
        lo = max(0, int(start))
        hi = min(height * width, int(start + length))
        if lo < hi:
            mask[lo:hi] = 1
    return mask.reshape((height, width), order="F")

print("Dataset loaded:", train.shape)
print("Images:", train["image_id"].nunique())
print("Classes:", sorted(train["defect_type"].unique()))
print("Train directory exists:", TRAIN_DIR.exists())



Dataset loaded: (45572, 5)
Images: 11393
Classes: ['defect_a', 'defect_b', 'defect_c', 'defect_d']
Train directory exists: True


**MILESTONE 1**

In [4]:
#Q1
positive = train["mask_rle"].ne("")

class_counts = (
    train.assign(positive=positive)
    .groupby("defect_type")["positive"]
    .sum()
    .astype(int)
    .sort_values(ascending=False)
)

positive_per_image = (
    train.assign(positive=positive)
    .groupby("image_id")["positive"]
    .sum()
)

clean_images = int((positive_per_image == 0).sum())
most_frequent_count = int(class_counts.max())
answer_q1 = most_frequent_count + clean_images

print("Positive images per class:")
print(class_counts)
print("Completely clean images:", clean_images)
print("Most frequent class count:", most_frequent_count)
print("Q1 answer:", answer_q1)


Positive images per class:
defect_type
defect_c    4100
defect_d    1707
defect_a    1081
defect_b     313
Name: positive, dtype: int64
Completely clean images: 5516
Most frequent class count: 4100
Q1 answer: 9616


In [5]:
#Q2

positive = train["mask_rle"].ne("")

labels_per_image = (
    train.assign(positive=positive)
    .pivot_table(
        index="image_id",
        columns="defect_type",
        values="positive",
        aggfunc="max",
        fill_value=False
    )
    .reindex(columns=CLASSES, fill_value=False)
)

cardinality = labels_per_image.sum(axis=1)

print("Images with 0 defects:", int((cardinality == 0).sum()))
print("Images with 1 defect:", int((cardinality == 1).sum()))
print("Images with 2 defects:", int((cardinality == 2).sum()))
print("Images with 3 defects:", int((cardinality == 3).sum()))
print("Images with 4 defects:", int((cardinality == 4).sum()))

answer_q2 = int((cardinality > 1).sum())
print("Q2 answer:", answer_q2)


Images with 0 defects: 5516
Images with 1 defect: 4679
Images with 2 defects: 1072
Images with 3 defects: 126
Images with 4 defects: 0
Q2 answer: 1198


In [6]:
#Q3

overlap_pixels = 0
multi_label_ids = cardinality[cardinality > 1].index

for image_id in multi_label_ids:
    rows = train[train["image_id"] == image_id]
    label_count = np.zeros((HEIGHT, WIDTH), dtype=np.uint8)

    for rle in rows["mask_rle"]:
        if rle:
            label_count += decode_rle(rle)

    overlap_pixels += int((label_count >= 2).sum())

print("Multi-label images examined:", len(multi_label_ids))
print("Q3 answer — overlapping pixels:", overlap_pixels)


Multi-label images examined: 1198
Q3 answer — overlapping pixels: 0


In [7]:
#Q4

malformed_rows = 0
max_pixel_index = 0
malformed_examples = []

for row_idx, rle in enumerate(train["mask_rle"]):
    if not rle:
        continue

    values = np.fromstring(rle, sep=" ", dtype=np.int64)
    if len(values) % 2:
        raise ValueError(f"Odd RLE value count in row {row_idx}")

    starts = values[0::2]
    lengths = values[1::2]
    ends = starts + lengths - 1

    if len(ends):
        max_pixel_index = max(max_pixel_index, int(ends.max()))

    bad = ends > N_PIXELS
    if bad.any():
        malformed_rows += 1
        if len(malformed_examples) < 5:
            malformed_examples.append({
                "row": row_idx,
                "max_end": int(ends.max()),
                "bad_runs": int(bad.sum())
            })

print(f"Q4 answer: {malformed_rows} rows, max index {max_pixel_index}")
print("Example malformed rows:", malformed_examples)


Q4 answer: 0 rows, max index 102400
Example malformed rows: []


In [8]:
#Q5

native_height, native_width = 128, 800
target_height, target_width = 256, 256

sx = target_width / native_width
sy = target_height / native_height
ratio = sy / sx

print(f"s_x = {sx:.4f}")
print(f"s_y = {sy:.4f}")
print(f"s_y / s_x = {ratio:.4f}")
print(f"Q5 answer: {sx:.4f}, {sy:.4f}, {ratio:.4f}")


s_x = 0.3200
s_y = 2.0000
s_y / s_x = 6.2500
Q5 answer: 0.3200, 2.0000, 6.2500


In [15]:
#Q6


import numpy as np
import pandas as pd
from PIL import Image
from scipy import ndimage

HEIGHT, WIDTH = 128, 800
CLASSES = ["defect_a", "defect_b", "defect_c", "defect_d"]
rng = np.random.default_rng(42)

def decode_rle(rle, height=HEIGHT, width=WIDTH):
    mask = np.zeros(height * width, dtype=np.uint8)
    if not str(rle).strip():
        return mask.reshape(height, width, order="F")
    values = np.fromstring(str(rle), sep=" ", dtype=np.int64)
    starts = values[0::2] - 1
    lengths = values[1::2]
    for start, length in zip(starts, lengths):
        if start < 0 or length < 0 or start + length > mask.size:
            raise ValueError("Invalid RLE")
        mask[start:start + length] = 1
    return mask.reshape(height, width, order="F")

def image_path(image_id):
    for ext in [".jpg", ".jpeg", ".png"]:
        path = TRAIN_DIR / f"{image_id}{ext}"
        if path.exists():
            return path
    raise FileNotFoundError(f"Image not found: {image_id}")

def cohens_d(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    if len(a) < 2 or len(b) < 2:
        return np.nan
    pooled_var = (
        (len(a)-1)*np.var(a, ddof=1)
        + (len(b)-1)*np.var(b, ddof=1)
    ) / (len(a)+len(b)-2)
    if pooled_var <= 0:
        return np.nan
    return (np.mean(a)-np.mean(b)) / np.sqrt(pooled_var)

results = []

for defect in CLASSES:
    fg_values, bg_values = [], []
    rows = train[
        (train["defect_type"] == defect)
        & train["mask_rle"].astype(str).str.strip().ne("")
    ]

    for row in rows.itertuples(index=False):
        with Image.open(image_path(str(row.image_id))) as im:
            gray = np.asarray(im.convert("L"))

        mask = decode_rle(row.mask_rle).astype(bool)
        if gray.shape != mask.shape or not mask.any():
            continue

        ring = ndimage.binary_dilation(
            mask, structure=np.ones((11, 11), dtype=bool)
        ) & ~mask

        fg, bg = gray[mask], gray[ring]
        if len(fg) and len(bg):
            fg_values.extend(
                rng.choice(fg, min(len(fg), 5000), replace=False)
            )
            bg_values.extend(
                rng.choice(bg, min(len(bg), 5000), replace=False)
            )

    d = cohens_d(fg_values, bg_values)
    results.append({
        "defect_type": defect,
        "cohens_d": d,
        "absolute_d": abs(d) if np.isfinite(d) else np.nan
    })

results = pd.DataFrame(results)
print(results.to_string(index=False))

eligible = results[
    results["absolute_d"].notna()
    & (results["absolute_d"] <= 0.02)
]

if len(eligible):
    print("\nClass(es) meeting |Cohen's d| <= 0.02:")
    for defect in eligible["defect_type"]:
        print(f"{defect}, texture")
else:
    print("\nNo class met the threshold with this calculation.")
    print("Check the official guide's exact sampling procedure.")


defect_type  cohens_d  absolute_d
   defect_a -0.019833    0.019833
   defect_b -0.105154    0.105154
   defect_c  0.067178    0.067178
   defect_d -0.154918    0.154918

Class(es) meeting |Cohen's d| <= 0.02:
defect_a, texture


In [10]:
#Q7

def dice_from_counts(pred_pixels, true_pixels, intersection):
    denominator = pred_pixels + true_pixels
    if denominator == 0:
        return 1.0
    return 2.0 * intersection / denominator

dice1 = dice_from_counts(pred_pixels=0, true_pixels=0, intersection=0)
dice2 = dice_from_counts(pred_pixels=2, true_pixels=0, intersection=0)

print(f"Q7 answer: {dice1:.1f}, {dice2:.1f}")


Q7 answer: 1.0, 0.0


In [11]:
#Q8

foreground_dice = (6 * 0.80 + 4 * 0.40) / 10

canvas_pixels = 102400
false_positive_pairs = 4
false_pixels_per_pair = 1024
empty_pairs = 40
correctly_empty_pairs = 36

mean_fp_fraction = (
    false_positive_pairs * false_pixels_per_pair
    / canvas_pixels
    / empty_pairs
)
false_positive_score = 1 - mean_fp_fraction
empty_accuracy = correctly_empty_pairs / empty_pairs

final_score = (
    0.70 * foreground_dice
    + 0.20 * false_positive_score
    + 0.10 * empty_accuracy
)

print(f"ForegroundDice: {foreground_dice:.4f}")
print(f"FalsePositiveScore: {false_positive_score:.4f}")
print(f"EmptyAccuracy: {empty_accuracy:.4f}")
print(f"Final score: {final_score:.4f}")


ForegroundDice: 0.6400
FalsePositiveScore: 0.9990
EmptyAccuracy: 0.9000
Final score: 0.7378


In [12]:
#Q9

positive = train["mask_rle"].ne("")
n_pairs = len(train)
n_positive = int(positive.sum())
n_empty = n_pairs - n_positive

# Empty-aware mean Dice: 1 for empty GT, 0 for positive GT
score_naive = n_empty / n_pairs

# All-empty prediction:
# ForegroundDice = 0, FalsePositiveScore = 1, EmptyAccuracy = 1
foreground_dice = 0.0 if n_positive else 1.0
false_positive_score = 1.0
empty_accuracy = 1.0

score_official = (
    0.70 * foreground_dice
    + 0.20 * false_positive_score
    + 0.10 * empty_accuracy
)

print("Total image-class pairs:", n_pairs)
print("Empty ground-truth pairs:", n_empty)
print("Positive ground-truth pairs:", n_positive)
print(f"Naive mean Dice: {score_naive:.4f}")
print(f"Official weighted score: {score_official:.4f}")
print(f"Q9 answer: {score_naive:.4f}, {score_official:.4f}")


Total image-class pairs: 45572
Empty ground-truth pairs: 38371
Positive ground-truth pairs: 7201
Naive mean Dice: 0.8420
Official weighted score: 0.3000
Q9 answer: 0.8420, 0.3000


In [13]:
#Q10
def component_statistics(defect_type, thresholds=(50, 200)):
    rows = train[
        (train["defect_type"] == defect_type)
        & train["mask_rle"].ne("")
    ]

    min_area = None
    total_pixels = 0
    lost_pixels = {n: 0 for n in thresholds}

    for rle in rows["mask_rle"]:
        mask = decode_rle(rle).astype(np.uint8)
        total_pixels += int(mask.sum())

        if not mask.any():
            continue

        n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
            mask, connectivity=8
        )
        areas = stats[1:, cv2.CC_STAT_AREA]
        if len(areas):
            current_min = int(areas.min())
            min_area = current_min if min_area is None else min(min_area, current_min)

            for n in thresholds:
                lost_pixels[n] += int(areas[areas < n].sum())

    lost_pct = {
        n: (100 * lost_pixels[n] / total_pixels if total_pixels else np.nan)
        for n in thresholds
    }
    return min_area, total_pixels, lost_pct

min_b, pixels_b, lost_b = component_statistics("defect_b", (50,))
_, pixels_a, lost_a = component_statistics("defect_a", (200,))

print("Smallest defect_b component:", min_b)
print("Total defect_b foreground pixels:", pixels_b)
print(f"defect_b pixels lost with N=50: {lost_b[50]:.6f}%")
print("Total defect_a foreground pixels:", pixels_a)
print(f"defect_a pixels lost with N=200: {lost_a[200]:.6f}%")
print(
    f"Q10 answer: {min_b}, {lost_b[50]:.6f}, {lost_a[200]:.6f}"
)


Smallest defect_b component: 93
Total defect_b foreground pixels: 215701
defect_b pixels lost with N=50: 0.000000%
Total defect_a foreground pixels: 809388
defect_a pixels lost with N=200: 15.217053%
Q10 answer: 93, 0.000000, 15.217053


In [14]:
#Q11

presence = (
    train.assign(positive=train["mask_rle"].ne(""))
    .pivot_table(
        index="image_id",
        columns="defect_type",
        values="positive",
        aggfunc="max",
        fill_value=False
    )
    .reindex(columns=CLASSES, fill_value=False)
    .astype(bool)
)

def combination_string(row):
    labels = [c for c in CLASSES if row[c]]
    return "+".join(labels) if labels else "clean"

presence["combination"] = presence.apply(combination_string, axis=1)

n_combos = int(presence["combination"].nunique())
b_positive_images = int(presence["defect_b"].sum())
b_in_val = int(np.floor(0.20 * b_positive_images + 0.5))

print("Unique combination strings:", n_combos)
print("defect_b-positive images:", b_positive_images)
print("Ideal fractional validation allocation:", 0.20 * b_positive_images)
print("Rounded validation count:", b_in_val)
print(f"Q11 answer: {n_combos}, {b_in_val}")


Unique combination strings: 15
defect_b-positive images: 313
Ideal fractional validation allocation: 62.6
Rounded validation count: 63
Q11 answer: 15, 63
